In [1]:
# imports
import altair as alt
alt.data_transformers.disable_max_rows()
import polars as pl
from pathlib import Path
from vega_datasets import data
import geopandas as gpd
import os, sys
import json
sys.path.insert(0, os.path.abspath(".."))

#theme and colors
from src.theme import lac_theme


In [2]:
migrants_from_LAC = pl.read_csv(Path("../Data/migrants_from_LAC.csv"))
migrants_grouped = pl.read_csv(Path("../Data/grouped_by_country.csv"))

In [3]:
def save_chart(chart, filename):
    """
    Save an Altair chart as an SVG file in the ../output_charts directory.
    Creates the folder if it does not exist.
    """
    name = f"../static-viz/{filename}.svg"
    print(f"Saving chart to: {name}")
    save_path = Path(name)
    os.makedirs(save_path.parent, exist_ok=True)
    chart.save(str(save_path), background="transparent")

In [4]:


def migration_trend(df):
    """
    Line chart showing total migration trend in Latin America and the Caribbean
    with clearly visible data labels.
    """

    line = alt.Chart(df).mark_line(
        color="#002147", 
        strokeWidth=3).encode(
        x=alt.X("year:N", title="Year"),
        y=alt.Y("sum(total_migrants):Q", title="Total Migrants (Millions)"))

    points = alt.Chart(df).mark_point(
        color="#002147", 
        size=50).encode(
        x="year:N",
        y="sum(total_migrants):Q")

    text = alt.Chart(df).mark_text(
        fontSize=10,
        fontWeight="bold",
        color="#002147",
        dy=-10,                
        align="center").encode(
        x="year:N",
        y="sum(total_migrants):Q",
        text=alt.Text("sum(total_migrants):Q", format=".2f"))

    chart = (
        (line + points + text)
        .properties(
            width=300,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text="Migration Trend in Latin America and the Caribbean",
                anchor="middle")))

    return chart

save_chart(migration_trend(migrants_grouped), "migration_trend_lac_labels")
migration_trend(migrants_grouped)

Saving chart to: ../static-viz/migration_trend_lac_labels.svg


/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/data.py:281: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


alt.LayerChart(...)

In [5]:
def migration_trend_mexico(df):
    """
    Line chart showing total migration trend in Latin America and the Caribbean
    with clearly visible data labels.
    """
    df= df.filter(pl.col("origin")=="Mexico")
    line = alt.Chart(df).mark_line(
        color="#fb923c", 
        strokeWidth=3).encode(
        x=alt.X("year:N", title="Year"),
        y=alt.Y("sum(total_migrants):Q", title="Total Migrants (Millions)"))

    points = alt.Chart(df).mark_point(
        color="#fb923c", 
        size=50).encode(
        x="year:N",
        y="sum(total_migrants):Q")
    
    text = alt.Chart(df).mark_text(
        fontSize=6,
        fontWeight="bold",
        color="#fb923c",
        dy=-15,              
        align="center").encode(
        x="year:N",
        y="sum(total_migrants):Q",
        text=alt.Text("sum(total_migrants):Q", format=".2f"))

    chart = (
        (line + points + text)
        .properties(
            width=200,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text="Migration Trend in Mexico",
                anchor="middle")))

    return chart

save_chart(migration_trend_mexico(migrants_grouped), "migration_trend_mexico_labels")
migration_trend_mexico(migrants_grouped)

Saving chart to: ../static-viz/migration_trend_mexico_labels.svg


alt.LayerChart(...)

In [6]:
def migration_trend_venezuela(df):
    """
    Line chart showing total migration trend in Latin America and the Caribbean
    with clearly visible data labels.
    """
    df= df.filter(pl.col("origin")=="Venezuela")
 
    line = alt.Chart(df).mark_line(
        color="#f87171", 
        strokeWidth=3).encode(
        x=alt.X("year:N", title="Year"),
        y=alt.Y("sum(total_migrants):Q", title="Total Migrants (Millions)"))

    points = alt.Chart(df).mark_point(
        color="#f87171", 
        size=50).encode(
        x="year:N",
        y="sum(total_migrants):Q")

    text = alt.Chart(df).mark_text(
        fontSize=6,
        fontWeight="bold",
        color="#f87171",
        dy=-15,              
        align="center").encode(
        x="year:N",
        y="sum(total_migrants):Q",
        text=alt.Text("sum(total_migrants):Q", format=".2f"))

    chart = (
        (line + points + text)
        .properties(
            width=200,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text="Migration Trend in Venezuela",
                anchor="middle")))

    return chart

save_chart(migration_trend_venezuela(migrants_grouped), "migration_trend_venezuela_labels")
migration_trend_venezuela(migrants_grouped)

Saving chart to: ../static-viz/migration_trend_venezuela_labels.svg


alt.LayerChart(...)

In [7]:
def migration_from(df):
    """
    Stacked bar chart showing the share of total migrants by subregion
    """
    bars = alt.Chart(df).mark_bar().encode(
        x=alt.X("year:N", title="Year"),
        y=alt.Y("sum(total_migrants):Q", title="Total Migrants (Millions)"),
        color=alt.Color("intermediate-region:N", title="")
    )

    chart = (
        bars
        .configure_legend(
            orient="top",                
            direction="horizontal",
            symbolType="square",
            columns=3,
            columnPadding=10,
            padding=1,                 
        )
        .properties(
            width=250,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text=["Migration by Sub-Region",
                      "grouped by sub-region of origin" ])))


    return chart

In [8]:
save_chart(migration_from(migrants_grouped), "migration_by_subregion")
migration_from(migrants_grouped)

Saving chart to: ../static-viz/migration_by_subregion.svg


alt.Chart(...)

In [9]:
def migration_from_norm_stacked(df):
    """
    100% stacked bar chart showing the share of total migrants by subregion,
    with correct in-bar percentage labels.
    """

    bars = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y(
                "sum(total_migrants):Q",
                title="Share of Total Migrants (%)",
                stack="normalize"
            ),
            color=alt.Color("intermediate-region:N", title="Subregion")
        )
    )


    chart = (
        bars
        .configure_legend(
            orient="top",                
            direction="horizontal",
            symbolType="square",
            columns=3,
            columnPadding=10,
            padding=1,                 
        )
        .properties(
            width=250,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text=["Migration by Sub-Region",
                      "grouped by sub-region of origin" ])))


    return chart


In [10]:
save_chart(migration_from_norm_stacked(migrants_grouped), "migration_from_stacked")
migration_from_norm_stacked(migrants_grouped)

Saving chart to: ../static-viz/migration_from_stacked.svg


alt.Chart(...)

In [11]:
# We want a chart that shows the GDP difference between origin and destination countries
migrants_grouped = migrants_grouped.with_columns([
    # Absolute difference (Destination − Origin)
    (pl.col("weighted_avg_dest_gdp") - pl.col("gdp_per_capita")).alias("gdp_difference"),
    
    # Relative difference in percent
    ((pl.col("weighted_avg_dest_gdp") - pl.col("gdp_per_capita"))
        / pl.col("gdp_per_capita") * 100).alias("gdp_difference_percent")])

# Lets double check the calculations
gdp_difference = migrants_grouped.filter(pl.col("year") == 2024)
gdp_difference.filter(pl.col("origin") == "Mexico")



origin,M49,alpha-2,alpha-3,region,sub-region,intermediate-region,year,total_migrants,female_migrants,male_migrants,pct_female_migrants,pct_male_migrants,remittance_inflows,gdp_per_capita,total_population,Gini_index,homicides_per_100k_population,exposure_to_droughts,climate_risk_index,migration_rate,weighted_avg_dest_gdp,gdp_difference,gdp_difference_percent
str,i64,str,str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Mexico""",484,"""MX""","""MEX""","""Americas""","""Latin America and the Caribbea…","""Central America""",2024,11.596529,5.665313,5.931216,48.853523,51.146477,null,14157.944584,130.861007,null,28.175653,null,112.0,8.861715,25322.255083,11164.3105,78.855447


In [12]:
def top_n_gdp_gap_2024(df,n=10):
    """
    Top n migration-origin countries in 2024 
    with their GDP per capita gap (destination vs origin).
    """

    top_ten= (
        df.filter(pl.col("year") == 2024)
          .drop_nulls(["total_migrants", "gdp_difference_percent"])
          .sort("total_migrants", descending=True)
          .head(n)
          .with_columns(
              pl.when(pl.col("gdp_difference_percent") > 0)
                .then(pl.lit("positive"))
                .otherwise(pl.lit("negative"))
                .alias("gap_sign"))
          .select([
              "origin",
              "sub-region",
              "total_migrants",
              "gdp_per_capita",
              "total_population",
              "weighted_avg_dest_gdp",
              "gdp_difference",
              "gdp_difference_percent",
              "gap_sign"]))

    return top_ten



In [13]:
def plot_top_n_gdp_gap(df):
    """
    Chart comparing GDP per capita between origin and destination
    for the top migration-origin countries in Latin America and the Caribbean (2024).
    """
    data = df.sort("total_migrants", descending=True)

    lines = (
        alt.Chart(data)
        .mark_rule(color="#9CA3AF", strokeWidth=2)
        .encode(
            x="gdp_per_capita:Q",
            x2="weighted_avg_dest_gdp:Q",
            y=alt.Y(
                "origin:N",
                sort=alt.SortField(field="total_migrants", order="descending"),
                title="Top 10 Origin Countries (2024)")))

    origin_points = (
        alt.Chart(data)
        .mark_point(color="#002147", size=70)
        .encode(
            x=alt.X("gdp_per_capita:Q", title="GDP per Capita (USD)"),
            y=alt.Y(
                "origin:N",
                sort=alt.SortField(field="total_migrants", order="descending")
            )))

    dest_points = (
        alt.Chart(data)
        .mark_point(color="#DC2626", size=70)
        .encode(
            x="weighted_avg_dest_gdp:Q",
            y=alt.Y(
                "origin:N",
                sort=alt.SortField(field="total_migrants", order="descending"))))

    chart = (
        (lines + origin_points + dest_points)
        .properties(
            width=520,
            height=700,
            background="transparent",
            title=alt.TitleParams(
                text="Gap in GDP per Capita between Origin and Destination (2024)",
                subtitle="Top 10 Latin American and Caribbean countries by migrant stock",
                anchor="middle")))

    return chart


gdp_dumbell = plot_top_n_gdp_gap(top_n_gdp_gap_2024(migrants_grouped, n=20))
save_chart(gdp_dumbell, "top10_gdp_gap_dumbbell_2024")
gdp_dumbell


Saving chart to: ../static-viz/top10_gdp_gap_dumbbell_2024.svg


alt.LayerChart(...)

In [14]:
def rank_data(df):
    """
    Rank data for the top 5 origin countries by total migrants per year.
    """

    ranked = (
        df.filter(pl.col("year") >= 1990)
          .group_by(["year", "origin"])
          .agg(pl.col("total_migrants").sum().alias("total"))
          .with_columns(
              pl.col("total")
              .rank("dense", descending=True)
              .over("year")
              .alias("rank")
          )
          .filter(pl.col("rank") <= 5)
          .sort(["year", "rank"])
    )
    return ranked


def rank_line(df):
    """
    Create a rank line (bump chart) for the top 5 origin countries over time.
    Transparent background, Oxford Blue palette, and labels for 2024.
    """

    lines = (alt.Chart(df)
        .mark_line(point=True, strokeWidth=3)
        .encode(
            x=alt.X("year:O", title="Year"),
            y=alt.Y("rank:Q",
                    title="Rank (1 = Highest)",
                    scale=alt.Scale(reverse=True)),
            color=alt.Color("origin:N", title="Origin Country")) )

    labels = (
        alt.Chart(df.filter(pl.col("year") == 2024))
        .mark_text(align="left", fontWeight="bold")
        .encode(
            x=alt.X("year:O"),
            y=alt.Y("rank:Q", scale=alt.Scale(reverse=True)),
            text="origin:N",
            color=alt.Color("origin:N")))
    
    chart = (
        (lines + labels)
        .properties(
            width=450,
            height=200,
            background="transparent",
            title=alt.TitleParams(
                text="Top 5 Origin Countries by Total Migrants (1990–2024)",
                anchor="middle")))

    return chart


# === Example usage ===
origins_rank = rank_data(migrants_grouped)
chart_origins = rank_line(origins_rank)
save_chart(chart_origins, "rank_line_origins")
chart_origins


Saving chart to: ../static-viz/rank_line_origins.svg


alt.LayerChart(...)

In [15]:
def get_latest_dataset(variable, df= migrants_grouped):
    """
    Returns the latest available data per country for a chosen variable,
    including migration and demographic context columns.
    """
    keep_cols = [
        "origin", "year", "region", "sub-region",  "intermediate-region","alpha-3", variable,
        "total_migrants", "female_migrants", "male_migrants",
        "pct_female_migrants", "pct_male_migrants", "total_population", "migration_rate"
    ]

    # Make sure we only select columns that exist in the DataFrame
    keep_cols = [col for col in keep_cols if col in df.columns]

    # Sort and keep the most recent observation per country
    latest = (
        df
        .select(keep_cols)
        .sort(["origin", "year"], descending=[False, True])
        .unique(subset=["origin"], keep="first")
        .drop_nulls(variable)
    )

    return latest

# an example to see if it works
get_latest_dataset("gdp_per_capita")

origin,year,region,sub-region,intermediate-region,alpha-3,gdp_per_capita,total_migrants,female_migrants,male_migrants,pct_female_migrants,pct_male_migrants,total_population,migration_rate
str,i64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""Antigua and Barbuda""",2024,"""Americas""","""Latin America and the Caribbea…","""Caribbean""","""ATG""",23725.790373,0.013111,0.006705,0.006406,51.140264,48.859736,0.093772,13.981786
"""Argentina""",2024,"""Americas""","""Latin America and the Caribbea…","""South America""","""ARG""",13858.20398,1.183381,0.592218,0.591163,50.044576,49.955424,45.696159,2.589673
"""Bahamas""",2024,"""Americas""","""Latin America and the Caribbea…","""Caribbean""","""BHS""",39455.446655,0.002174,0.001088,0.001086,50.045998,49.954002,0.401283,0.541762
"""Barbados""",2024,"""Americas""","""Latin America and the Caribbea…","""Caribbean""","""BRB""",25365.794942,0.019558,0.010083,0.009475,51.554351,48.445649,0.282467,6.923995
"""Belize""",2024,"""Americas""","""Latin America and the Caribbea…","""Central America""","""BLZ""",8429.679597,0.007383,0.003774,0.003609,51.117432,48.882568,0.417072,1.770198
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Sint Maarten (Dutch part)""",2024,"""Americas""","""Latin America and the Caribbea…","""Caribbean""","""SXM""",40027.917599,0.002464,0.001257,0.001207,51.01461,48.98539,0.04335,5.683968
"""Suriname""",2024,"""Americas""","""Latin America and the Caribbea…","""South America""","""SUR""",7430.702192,0.258026,0.14465,0.113376,56.060242,43.939758,0.634431,40.670459
"""Trinidad and Tobago""",2024,"""Americas""","""Latin America and the Caribbea…","""Caribbean""","""TTO""",19314.716343,0.32363,0.183901,0.139729,56.82446,43.17554,1.368333,23.651406


In [16]:
def chart_scatter_vs_migration(variable):
    """
    Creates a scatter plot comparing any variable to migration_rate,
    using the latest available data for each country from a Polars DataFrame.
    Adds a smooth LOESS regression line to illustrate the overall trend.
    """

    latest = get_latest_dataset(variable)



    scatter = (
        alt.Chart(latest)
        .mark_point(filled=True, color="#1E3A8A", opacity=0.7, size=80)
        .encode(
            x=alt.X(f"{variable}:Q", title=variable.replace("_", " ").title()),
            y=alt.Y("migration_rate:Q", title="Migration Rate (%)")))

    line = (
        alt.Chart(latest)
        .transform_loess(f"{variable}", "migration_rate", bandwidth=0.5)
        .mark_line(color="#DC2626")
        .encode(x=f"{variable}:Q", y="migration_rate:Q"))

    chart = ((scatter + line)
        .properties(
            title=alt.TitleParams(
                text=f"{variable.replace('_', ' ').title()} vs Migration Rate",
                anchor="middle"),
            width=300,
            height=420,
            background="transparent" ))

    return chart



In [17]:
gini_index= chart_scatter_vs_migration("Gini_index")
save_chart(gini_index, "scatter_gini_vs_migration")
gini_index



Saving chart to: ../static-viz/scatter_gini_vs_migration.svg


alt.LayerChart(...)

In [18]:
homicides = chart_scatter_vs_migration("homicides_per_100k_population")
save_chart(homicides, "homicides_vs_migration_rate")
homicides



Saving chart to: ../static-viz/homicides_vs_migration_rate.svg


alt.LayerChart(...)

In [19]:

climate_risk = chart_scatter_vs_migration("climate_risk_index")
save_chart(climate_risk, "climate_risk_vs_migration_rate")
climate_risk


Saving chart to: ../static-viz/climate_risk_vs_migration_rate.svg


alt.LayerChart(...)

In [20]:
# shapefiles
gdf_ne = gpd.read_file(Path("../Data/un_shapefiles.zip"))[["geometry", "SOV_A3"]].rename(columns={"SOV_A3": "alpha-3"})
geo_df = gdf_ne[["alpha-3", "geometry"]].drop_duplicates(subset=["alpha-3"])
grouped_LAC_pd = migrants_grouped.to_pandas()
merged_df = grouped_LAC_pd.merge(geo_df, on="alpha-3", how="left")
gdf_LAC = gpd.GeoDataFrame(merged_df, geometry="geometry", crs="EPSG:4326")
gdf_LAC = gdf_LAC.sort_values(["origin", "year"])

In [21]:
def plot_lac_migration(gdf_LAC, year: int):
    """
    Create a transparent migration bubble map for Latin America and the Caribbean.
      - Bubble size = total migrants
      - Color = intermediate region
      - Excludes Puerto Rico & U.S. Virgin Islands
    """

    gdf_lac = (
        gdf_LAC[
            (gdf_LAC["region"] == "Americas") &
            (gdf_LAC["sub-region"] == "Latin America and the Caribbean") &
            (gdf_LAC["year"] == year) &
            (~gdf_LAC["origin"].isin([]))
        ]
        .dropna(subset=["origin", "total_migrants","geometry"])
        .to_crs("EPSG:4326")
        .copy()
    )

    gdf_lac["lon"] = gdf_lac.geometry.centroid.x
    gdf_lac["lat"] = gdf_lac.geometry.centroid.y

    geojson = json.loads(gdf_lac.to_json())

    basemap = (
        alt.Chart(alt.Data(values=geojson["features"]))
        .mark_geoshape(
            fill="#f3f4f6",
            stroke="#280b0bff",
            strokeWidth=1
        )
        .project(type="mercator")
        .properties(width=700, height=420)
    )

    bubbles = (
        alt.Chart(gdf_lac)
        .mark_circle(opacity=0.85, stroke="#1F2937", strokeWidth=0.25)
        .encode(
            longitude="lon:Q",
            latitude="lat:Q",
            size=alt.Size(
                "total_migrants:Q",
                title="Total Million Migrants",
                scale=alt.Scale(range=[30, 1500]),
                legend=alt.Legend(
                    symbolType="circle")),
            color=alt.Color(
                "intermediate-region:N",
                title="",
                scale=alt.Scale(
                    domain=["Caribbean", "Central America", "South America"],
                    range=["#60A5FA", "#FB923C", "#F87171"]))))

    chart = ((basemap + bubbles)
        .properties(
            title=alt.TitleParams(
                text=[
                    "Migration Stock from Latin America and the Caribbean",
                    f"Year: {year} ",]),
            background="transparent",
            width=350))

    return chart


In [22]:
lac_map = plot_lac_migration(gdf_LAC = gdf_LAC, year=2024)
save_chart(lac_map, "lac_migration_map_2024")
lac_map

Saving chart to: ../static-viz/lac_migration_map_2024.svg


/var/folders/2l/np27k3x96zlgg9n3dvyy9hxh0000gn/T/ipykernel_40518/3039035673.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_lac["lon"] = gdf_lac.geometry.centroid.x
/var/folders/2l/np27k3x96zlgg9n3dvyy9hxh0000gn/T/ipykernel_40518/3039035673.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_lac["lat"] = gdf_lac.geometry.centroid.y
/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/danielm/Desktop/mig

alt.LayerChart(...)

In [23]:
save_chart(lac_map, "net_migration_stock_2024")

/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list

Saving chart to: ../static-viz/net_migration_stock_2024.svg


In [24]:
def plot_lac_migration(gdf_LAC, year=2024):
    """
    Create  migration bubble map for Latin America and the Caribbean.
        Bubble size = total population
        Color gradient = share of migrants (% of population)
    """

    gdf_lac = (
        gdf_LAC
        .query("region == 'Americas' and `sub-region` == 'Latin America and the Caribbean'")
        .query(f"year == {year}")
        .dropna(subset=["origin", "total_population", "total_migrants", "migration_rate"])
        .to_crs("EPSG:4326")
        .copy())

    gdf_lac["lon"] = gdf_lac.geometry.centroid.x
    gdf_lac["lat"] = gdf_lac.geometry.centroid.y

    geojson = json.loads(gdf_lac.to_json())

    basemap = (
        alt.Chart(alt.Data(values=geojson["features"]))
        .mark_geoshape(
            fill="#f3f4f6",
            stroke="#280b0bff",
            strokeWidth=1)
        .properties(width=700, height=420))

    bubbles = (
        alt.Chart(gdf_lac)
        .mark_circle(opacity=0.75, stroke="black", strokeWidth=0.2)
        .encode(
            longitude="lon:Q",
            latitude="lat:Q",
            size=alt.Size(
                "total_population:Q",
                title="Total Population",
                scale=alt.Scale(domain=[0.1, gdf_lac["total_population"].max()],
                                range=[30, 1800])),
            color=alt.Color(
                "migration_rate:Q",
                title="% of Migrants",
                scale=alt.Scale(
                    scheme="blues",
                    domain=[0, gdf_lac["migration_rate"].max()]))))


    chart = ((basemap + bubbles)
        .properties(
            title={
                "text": f"Migration Dynamics in Latin America and the Caribbean ({year})",
                "subtitle": [
                    "Bubble size = Total population",
                    "Color gradient = % of migrants"],
                "anchor": "start"}))

    return chart


In [25]:
proportional_migrationmap=plot_lac_migration(gdf_LAC, year=2024)
save_chart(proportional_migrationmap, "proportional_migration_map_2024")
proportional_migrationmap

Saving chart to: ../static-viz/proportional_migration_map_2024.svg


/var/folders/2l/np27k3x96zlgg9n3dvyy9hxh0000gn/T/ipykernel_40518/1549099079.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_lac["lon"] = gdf_lac.geometry.centroid.x
/var/folders/2l/np27k3x96zlgg9n3dvyy9hxh0000gn/T/ipykernel_40518/1549099079.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_lac["lat"] = gdf_lac.geometry.centroid.y
/Users/danielm/Desktop/migration_datavis/env/lib/python3.13/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Users/danielm/Desktop/mig

alt.LayerChart(...)